In [ ]:
# Chest X-Ray Dataset Preprocessing

# This notebook handles dataset cleaning and preprocessing for chest X-ray classification.

# Tasks:
# - Dataset loading
# - Label cleaning
#- Multi-label formatting
#- Train/validation split- Image preprocessing pipeline


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
import os
import pandas as pd

IMAGE_ROOT = "/content/drive/MyDrive/DS3 (2)/archive/images/images_normalized"


In [ ]:
df_images = pd.DataFrame({"filename": image_files})
csv_path = "/content/drive/MyDrive/available_images.csv"
df_images.to_csv(csv_path, index=False)

print("Saved:", csv_path)


Saved: /content/drive/MyDrive/available_images.csv


In [ ]:
#removing frontal and unwanted images from dataset

In [ ]:
out_path = "/content/drive/MyDrive/indiana_frontal_with_reports.csv"
df_final.to_csv(out_path, index=False)

print("Saved:", out_path)


Saved: /content/drive/MyDrive/indiana_frontal_with_reports.csv


In [ ]:
#creating 7 labeled dataset from reports

In [ ]:
import pandas as pd
import re

csv_path = "/content/drive/MyDrive/indiana_frontal_with_reports.csv"
df = pd.read_csv(csv_path)

print("Total samples:", len(df))
df.head()


Total samples: 3818


,uid,filename,projection,findings,impression
0,1,1_IM-0001-4001.dcm.png,Frontal,The cardiac silhouette and mediastinum size ar...,Normal chest x-XXXX.
1,2,2_IM-0652-1001.dcm.png,Frontal,Borderline cardiomegaly. Midline sternotomy XX...,No acute pulmonary findings.
2,3,3_IM-1384-1001.dcm.png,Frontal,NaN,"No displaced rib fractures, pneumothorax, or p..."
3,4,4_IM-2050-1001.dcm.png,Frontal,There are diffuse bilateral interstitial and a...,1. Bullous emphysema and interstitial fibrosis...
4,5,5_IM-2117-1003002.dcm.png,Frontal,The cardiomediastinal silhouette and pulmonary...,No acute cardiopulmonary abnormality.


In [ ]:
def combine_text(row):
    if isinstance(row["findings"], str) and row["findings"].strip():
        return row["findings"]
    elif isinstance(row["impression"], str):
        return row["impression"]
    else:
        return ""

df["report_text"] = df.apply(combine_text, axis=1)



In [ ]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["report_text"] = df["report_text"].apply(clean_text)


In [ ]:
KEYWORDS = {
    "lung_opacity": [
        "opacity", "opacities", "infiltrate", "infiltration", "airspace disease"
    ],
    "consolidation": [
        "consolidation", "airspace consolidation"
    ],
    "pleural_effusion": [
        "pleural effusion", "effusions"
    ],
    "cardiomegaly": [
        "cardiomegaly", "enlarged heart", "cardiac enlargement"
    ],
    "atelectasis": [
        "atelectasis", "collapse", "hypoinflation"
    ],
    "edema": [
        "edema", "pulmonary edema", "interstitial edema"
    ],
    "support_devices": [
        "tube", "line", "catheter", "picc", "endotracheal", "chest tube"
    ]
}


In [ ]:
def extract_labels(text):
    labels = {}
    for label, keywords in KEYWORDS.items():
        labels[label] = int(any(k in text for k in keywords))
    return pd.Series(labels)

labels_df = df["report_text"].apply(extract_labels)


In [ ]:
final_df = pd.concat(
    [df[["filename"]], labels_df],
    axis=1
)

out_path = "/content/drive/MyDrive/final_training_labels.csv"
final_df.to_csv(out_path, index=False)

print("Saved final labels CSV:", out_path)
final_df.head()


Saved final labels CSV: /content/drive/MyDrive/final_training_labels.csv


,filename,lung_opacity,consolidation,pleural_effusion,cardiomegaly,atelectasis,edema,support_devices
0,1_IM-0001-4001.dcm.png,0,1,1,0,0,1,0
1,2_IM-0652-1001.dcm.png,0,0,0,1,0,0,1
2,3_IM-1384-1001.dcm.png,0,0,1,0,0,0,0
3,4_IM-2050-1001.dcm.png,1,0,1,0,0,0,0
4,5_IM-2117-1003002.dcm.png,0,1,1,0,0,0,0
